### SETUP
Utilizaremos el conjunto de datos de entrenamiento ISIC 2019 y sus etiquetas, y que genere dos subconjuntos de datos a
partir de dicho conjunto: el primero (será el conjunto de entrenamiento de nuestro modelo) contendrá 100 imágenes de cada clase, mientras que el segundo (será el conjunto de test de nuestro modelo) contendrá 10 imágenes de cada clase.

In [ ]:
# Importaciones
import os
import shutil
import pandas as pd
import zipfile
import requests
from sklearn.model_selection import train_test_split

### Configuración

In [ ]:
# --- CONFIGURACIÓN INTELIGENTE ---
# Nombre de la carpeta donde debería estar el ZIP o las imágenes descomprimidas
NOMBRE_CARPETA_DATOS = "ISIC_2019_Training_Input"
NOMBRE_ZIP = "ISIC_2019_Training_Input.zip"

# URLs
URL_DATA = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
URL_LABELS = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv"

# Directorios de destino final
BASE_DIR = "dataset_isic"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
csv_path = "ground_truth.csv"

# Crear carpetas
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

# --- LÓGICA AUTOMÁTICA ---
print(" Verificando datos...")

# 1. Descargar CSV si no existe
if not os.path.exists(csv_path):
    print(" Descargando etiquetas (CSV)...")
    r = requests.get(URL_LABELS)
    with open(csv_path, 'wb') as f:
        f.write(r.content)

# 2. Buscar datos (Prioridad: Carpeta Local > Zip Local > Descargar)
origen_datos = None
tipo_origen = None

if os.path.exists(NOMBRE_CARPETA_DATOS):
    print(f" Carpeta de imágenes detectada localmente: {NOMBRE_CARPETA_DATOS}")
    origen_datos = NOMBRE_CARPETA_DATOS
    tipo_origen = "carpeta"
elif os.path.exists(NOMBRE_ZIP):
    print(f" ZIP detectado localmente: {NOMBRE_ZIP}")
    origen_datos = NOMBRE_ZIP
    tipo_origen = "zip"
else:
    print(" No se encuentran datos locales. Iniciando descarga (esto tardará)...")
    # Aquí iría el código de descarga del ZIP si fuera necesario
    # Para tu caso, mejor que tus compañeros se copien el zip a mano para no esperar
    print(" ERROR PARA EL GRUPO: Por favor, poned el archivo 'ISIC_2019_Training_Input.zip' en la misma carpeta que este notebook.")
    # Si quieres que se descargue solo, descomenta la función de descarga que te di al inicio.

# A partir de aquí sigue el código de extracción que ya tienes, 
# usando 'origen_datos' y 'tipo_origen' para decidir si copiar o descomprimir.

## Descarga y extracción de datos

In [ ]:
# Verificación
if not os.path.exists(csv_path) or not os.path.exists(zip_path):
    print("Error: Faltan archivos. Ejecuta la celda anterior.")
else:
    print("Archivos encontrados. Iniciando extracción selectiva...")

    # Cargar etiquetas
    df = pd.read_csv(csv_path)
    
    # Obtener clases (Ignorando 'image' y 'UNK')
    clases = [c for c in df.columns if c not in ['image', 'UNK']]
    print(f"Clases detectadas: {clases}")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # OPTIMIZACIÓN: Crear un mapa {nombre_archivo: ruta_completa_en_zip}
        # Esto hace que la búsqueda sea instantánea en lugar de buscar en bucle
        print("Mapeando contenido del ZIP (esto toma unos segundos)...")
        all_files_map = {os.path.basename(f): f for f in zip_ref.namelist()}
        
        for clase in clases:
            print(f"\nProcesando clase: {clase}")
            
            # Crear carpetas destino
            path_train = os.path.join(TRAIN_DIR, clase)
            path_test = os.path.join(TEST_DIR, clase)
            os.makedirs(path_train, exist_ok=True)
            os.makedirs(path_test, exist_ok=True)

            # Filtrar imágenes de la clase
            imgs_clase = df[df[clase] == 1.0]['image'].tolist()
            
            # Muestreo 100 train + 10 test
            if len(imgs_clase) < 110:
                seleccion = imgs_clase
            else:
                seleccion = df[df[clase] == 1.0]['image'].sample(n=110, random_state=42).tolist()
            
            lista_train = seleccion[:100]
            lista_test = seleccion[100:110]

            def extraer_lista(lista, carpeta_destino):
                count = 0
                for img_name in lista:
                    filename = img_name + ".jpg"
                    
                    # Buscamos en el mapa optimizado
                    ruta_en_zip = all_files_map.get(filename)
                    
                    if ruta_en_zip:
                        # Extraer directamente del zip al destino sin descomprimir todo
                        source = zip_ref.open(ruta_en_zip)
                        target = open(os.path.join(carpeta_destino, filename), "wb")
                        with source, target:
                            shutil.copyfileobj(source, target)
                        count += 1
                return count

            n_train = extraer_lista(lista_train, path_train)
            n_test = extraer_lista(lista_test, path_test)
            
            print(f"   -> Train: {n_train} | Test: {n_test}")

print("\n ¡Dataset preparado! Solo has extraído lo necesario.")

### Procesamiento de datos y creación de conjuntos de entrenamiento y test

In [ ]:
# Cargar el CSV
df = pd.read_csv(csv_path)

# Las clases son todas las columnas menos la primera ('image') y la última ('UNK') si existe
clases = df.columns[1:-1] if 'UNK' in df.columns else df.columns[1:]
print(f"Clases detectadas: {list(clases)}")

# TODO: Lógica principal de movimiento de archivos
for clase in clases:
    print(f"\nProcesando clase: {clase}")

    # Crear carpetas específicas para YOLO (dataset/train/MEL, dataset/test/MEL, etc.)
    os.makedirs(os.path.join(TRAIN_DIR, clase), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, clase), exist_ok=True)

    # Filtrar imágenes que pertenecen a esta clase (valor == 1.0)
    imagenes_clase = df[df[clase] == 1.0]['image'].tolist()

    # [cite_start]Seleccionar aleatoriamente 110 imágenes (100 train + 10 test) [cite: 27]
    if len(imagenes_clase) < 110:
        print(f" ADVERTENCIA: La clase {clase} tiene menos de 110 imágenes ({len(imagenes_clase)}). Se usarán todas.")
        seleccion = imagenes_clase
    else:
        # Mezclamos y cogemos 110
        seleccion = pd.Series(imagenes_clase).sample(n=110, random_state=42).tolist()

    # Dividir: Las primeras 100 para train, las siguientes 10 para test
    imgs_train = seleccion[:100]
    imgs_test = seleccion[100:110]

    # Función auxiliar para mover/copiar
    def mover_imagenes(lista_imgs, carpeta_destino):
        count = 0
        for img_name in lista_imgs:
            src = os.path.join(images_extract_path, img_name + ".jpg")
            dst = os.path.join(carpeta_destino, img_name + ".jpg")

            if os.path.exists(src):
                shutil.copy(src, dst)
                count += 1
        return count

    n_train = mover_imagenes(imgs_train, os.path.join(TRAIN_DIR, clase))
    n_test = mover_imagenes(imgs_test, os.path.join(TEST_DIR, clase))

    print(f" -> {n_train} imágenes movidas a TRAIN")
    print(f" -> {n_test} imágenes movidas a TEST")

print("\n Setup finalizado. Estructura de datos lista para YOLO.")